In [2]:
import pandas as pd
import numpy as np

In [3]:
# =======================
# FILE PATHS
# =======================

airbnb_path = "AIR_BNB_Data.xlsx"
travel_path = "Kaggle_Data.xlsx"
eia_path = "eia_Data.xlsx"
tsa_path = "TSA_Traveler_Throughput_Data.xlsx"
bls_path = "BLS_CPI_Data.xlsx"

# =======================
# LOAD DATA
# =======================

airbnb = pd.read_excel(airbnb_path)
travel = pd.read_excel(travel_path)
eia = pd.read_excel(eia_path)
tsa = pd.read_excel(tsa_path)
bls = pd.read_excel(bls_path)

# =======================
# CLEAN COLUMN NAMES
# =======================

def clean_cols(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
    )
    return df

airbnb = clean_cols(airbnb)
travel = clean_cols(travel)
eia = clean_cols(eia)
tsa = clean_cols(tsa)
bls = clean_cols(bls)

# =======================
# AIRBNB / LODGING
# =======================

airbnb["price"] = pd.to_numeric(airbnb["price"], errors = "coerce")

lodging_summary = (
    airbnb
    .groupby(["city", "room_type"], dropna=False)
    .agg(
        avg_nightly_price = ("price", "mean"),
        median_nighlty_price = ("price", "median"),
        listings = ("id", "count"),
        avg_availability=("availability_365", "mean")
    )
    .reset_index()
)

# =======================
# KAGGLE TRAVEL COST DATA
# =======================

travel["duration_days"] = pd.to_numeric(travel["duration_days"], errors="coerce")
travel["accommodation_cost"] = pd.to_numeric(travel["accommodation_cost"], errors="coerce")
travel["transportation_cost"] = pd.to_numeric(travel["transportation_cost"], errors="coerce")

travel["total_known_trip_cost"] = (
    travel["accommodation_cost"] + travel["transportation_cost"]
)

travel_summary = (
    travel
    .groupby("destination", dropna=False)
    .agg(
        avg_duration = ("duration_days", "mean"),
        avg_accommodation_cost = ("accommodation_cost", "mean"),
        avg_transportation_cost = ("transportation_cost", "mean"),
        avg_total_known_trip_cost = ("total_known_trip_cost", "mean"),
        trips = ("trip_id", "count")
    )
    .reset_index()
)

# =======================
# EIA GAS DATA
# =======================

eia_date_col = eia.columns[0]
eia[eia_date_col] = pd.to_datetime(eia[eia_date_col], errors="coerce")

eia_long = eia.melt(
    id_vars=eia_date_col,
    var_name="source_key",
    value_name="gas_price"
)

eia_long["gas_price"] = pd.to_numeric(eia_long["gas_price"], errors="coerce")
eia_long["year"] = eia_long[eia_date_col].dt.year

gas_summary = (
    eia_long
    .groupby(["year", "source_key"], dropna=False)
    .agg(avg_gas_price=("gas_price", "mean"))
    .reset_index()
)

# =======================
# TSA TRAVELER DATA
# =======================

tsa["date"] = pd.to_datetime(tsa["date"], errors="coerce")
tsa["numbers"] = pd.to_numeric(tsa["numbers"], errors="coerce")
tsa["year"] = tsa["date"].dt.year

tsa_summary = (
    tsa
    .groupby("year")
    .agg(
        avg_daily_travelers=("numbers", "mean"),
        total_travelers=("numbers", "sum")
    )
    .reset_index()
)

# =======================
# BLS BPI Data
# =======================

month_cols = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]

bls_long = bls.melt(
    id_vars=["series", "year"],
    value_vars=[c for c in month_cols if c in bls.columns],
    var_name="month",
    value_name="cpi_value"
)

bls_long["cpi_value"] = pd.to_numeric(bls_long["cpi_value"], errors="coerce")

bls_summary = (
    bls_long
    .groupby(["year", "series"], dropna=False)
    .agg(avg_cpi=("cpi_value", "mean"))
    .reset_index()
)

# =======================
# VACATION BUDGET SIMULATOR
# =======================

family_size = 4
vacation_days = 5

avg_daily_food_cost_per_person = 55
avg_activity_cost_per_person_per_day = 45

base_budget = travel_summary.copy()

base_budget["family_size"] = family_size
base_budget["vacation_days"] = vacation_days

base_budget["estimated_food_cost"] = (
    family_size * vacation_days * avg_daily_food_cost_per_person
)

base_budget["estimated_activity_cost"] = (
    family_size * vacation_days * avg_activity_cost_per_person_per_day
)

base_budget["estimated_total_vacation_cost"] = (
    base_budget["avg_accommodation_cost"]
    + base_budget["avg_transportation_cost"]
    + base_budget["estimated_food_cost"]
    + base_budget["estimated_activity_cost"]
)

base_budget["budget_level"] = pd.cut(
    base_budget["estimated_total_vacation_cost"],
    bins=[0, 2500, 5000, np.inf],
    labels=["Conservative", "Expected", "Comfortable"]
)

# =======================
# EXPORT CLEANED OUTPUTS
# =======================

lodging_summary.to_excel("lodge.xlsx", index=False)
travel_summary.to_excel("travel.xlsx", index=False)
gas_summary.to_excel("gas.xlsx", index=False)
tsa_summary.to_excel("air.xlsx", index=False)
bls_summary.to_excel("bls.xlsx", index=False)
base_budget.to_excel("vbs.xlsx", index=False)

print("Files created successfully.")

Files created successfully.
